# Lab 04 — Silver data quality and quarantine

This notebook converts the immutable Bronze records into a controlled Silver-ready candidate batch. It applies explicit business-quality rules, attaches all applicable rejection reasons, quarantines invalid records, and removes exact duplicates from the valid population.

## Objectives

- select one Bronze batch using `_batch_id`;
- evaluate every row against named quality rules;
- retain the first occurrence of an exact business duplicate and reject later occurrences;
- persist rejected rows and their reason arrays in a Delta quarantine table;
- normalize valid records into an explicit Silver candidate schema;
- write batch-level quality metrics idempotently;
- prove that total rows equal valid rows plus rejected rows.

> This notebook does **not** merge into the final Silver table. That responsibility belongs to `lab04_04_silver_merge.ipynb`.

## 1. Load shared configuration

In [0]:
%run ./lab04_00_config

In [0]:
import sys

from delta.tables import DeltaTable
from pyspark.sql import functions as F

lab04_root = (
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/labs/lab_04_silver_quality"
)

if lab04_root not in sys.path:
    sys.path.append(lab04_root)

from src.quality_rules import (
    EXPECTED_SOURCE_COLUMNS,
    apply_online_retail_quality_rules,
    build_quality_metrics,
    build_rule_failure_summary,
    split_valid_and_quarantine,
)

# Keep config and reusable module aligned.
if tuple(expected_source_columns) != tuple(EXPECTED_SOURCE_COLUMNS):
    raise AssertionError(
        "lab04_00_config expected_source_columns does not match "
        "src.quality_rules.EXPECTED_SOURCE_COLUMNS."
    )

silver_candidate_path = (
    f"{paths['landing']}/silver_candidates/batch_id={batch_id}"
)

print(f"Bronze source: {bronze_table}")
print(f"Selected batch: {batch_id}")
print(f"Contract version: {contract_version}")
print(f"Candidate path: {silver_candidate_path}")
print(f"Quarantine table: {quarantine_table}")
print(f"Quality metrics table: {quality_metrics_table}")
print("Reusable quality module: src.quality_rules")


## 2. Read and validate the selected Bronze batch

The batch filter prevents an incremental run from re-evaluating unrelated records. Before applying business rules, the notebook verifies that the Bronze table exists and contains the expected source and technical columns.

In [0]:
if not spark.catalog.tableExists(bronze_table):
    raise FileNotFoundError(
        f"Bronze table {bronze_table} does not exist. "
        "Run lab04_02_bronze_ingestion.ipynb first."
    )

bronze_df = spark.table(bronze_table)
required_bronze_columns = set(expected_source_columns) | {
    "_bronze_record_id",
    "_batch_id",
    "_record_hash",
    "_source_row_number",
    "_source_file",
    "_source_sheet",
    "_input_file_path",
    "_bronze_ingested_at",
}
missing_columns = sorted(required_bronze_columns - set(bronze_df.columns))
if missing_columns:
    raise AssertionError(f"Bronze is missing required columns: {missing_columns}")

batch_bronze_df = bronze_df.filter(F.col("_batch_id") == F.lit(batch_id))
bronze_batch_count = batch_bronze_df.count()
if bronze_batch_count == 0:
    available_batches = [row["_batch_id"] for row in bronze_df.select("_batch_id").distinct().collect()]
    raise ValueError(
        f"No Bronze rows found for batch_id={batch_id!r}. "
        f"Available batches: {sorted(available_batches)}"
    )

print(f"Bronze rows selected: {bronze_batch_count:,}")
display(batch_bronze_df.limit(20))

## 3. Apply reusable quality rules and identify duplicates

The data-quality implementation lives in `src/quality_rules.py`, so notebooks, Jobs, and tests use one shared rule definition.

Exact business duplicates are identified across the eight Online Retail source fields. The first deterministic occurrence is retained; later occurrences receive `DUPLICATE_BUSINESS_ROW`. Technical metadata is excluded from duplicate matching.


In [0]:
quality_df = apply_online_retail_quality_rules(
    batch_bronze_df,
    contract_version=contract_version,
    business_columns=EXPECTED_SOURCE_COLUMNS,
)

duplicate_row_count = quality_df.filter(
    F.col("_duplicate_rank") > 1
).count()

print(f"Exact duplicate rows to reject: {duplicate_row_count:,}")


## 4. Inspect named data-quality results

`src/quality_rules.py` attaches all applicable reason codes to `_quality_reasons`. A row can fail multiple checks, so the full array is retained.

| Rule code | Condition |
|---|---|
| `MISSING_INVOICE_NO` | Invoice number is null or blank |
| `MISSING_STOCK_CODE` | Product code is null or blank |
| `MISSING_DESCRIPTION` | Description is null or blank |
| `NON_POSITIVE_QUANTITY` | Quantity is null, zero, or negative |
| `MISSING_INVOICE_DATE` | Invoice timestamp is null |
| `NON_POSITIVE_UNIT_PRICE` | Price is null, zero, or negative |
| `MISSING_CUSTOMER_ID` | Customer ID is null or blank |
| `MISSING_COUNTRY` | Country is null or blank |
| `CANCELLED_INVOICE` | Invoice number begins with `C` |
| `DUPLICATE_BUSINESS_ROW` | Exact business duplicate after the first occurrence |


In [0]:
display(
    quality_df
    .groupBy("_quality_status")
    .agg(F.count("*").alias("rows"))
    .orderBy("_quality_status")
)

print(
    "✅ Quality rules applied through "
    "src.quality_rules.apply_online_retail_quality_rules()."
)


## 5. Inspect rule-level failures

Exploding the reason array produces one row per rule violation. Because one record can fail multiple rules, the sum of violations can be greater than the rejected-row count.

In [0]:
rule_failure_summary_df = build_rule_failure_summary(
    quality_df
)

display(rule_failure_summary_df)

display(
    quality_df
    .filter(F.col("_quality_status") == "REJECTED")
    .select(
        *EXPECTED_SOURCE_COLUMNS,
        "_quality_reasons",
        "_bronze_record_id",
    )
    .limit(50)
)


## 6. Split valid and rejected records

The split is exhaustive and mutually exclusive. A reconciliation assertion ensures that no Bronze row is lost or counted twice.

In [0]:
valid_quality_df, rejected_quality_df = (
    split_valid_and_quarantine(quality_df)
)

valid_count = valid_quality_df.count()
rejected_count = rejected_quality_df.count()

if valid_count + rejected_count != bronze_batch_count:
    raise AssertionError(
        f"Quality reconciliation failed: valid={valid_count}, "
        f"rejected={rejected_count}, Bronze={bronze_batch_count}."
    )

print(f"Bronze rows: {bronze_batch_count:,}")
print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {rejected_count:,}")
print("✅ Reconciliation passed: Bronze = valid + rejected.")


## 7. Persist rejected rows to the pre-created quarantine table

The quarantine table structure is created by `lab04_00_setup` outside the production Job.

Rejected rows are upserted by a stable quarantine key so retries remain idempotent. The Job notebook validates the target but performs no structural DDL.


In [0]:
quarantine_batch_df = (
    rejected_quality_df
    .withColumn(
        "_quarantine_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("_bronze_record_id"),
                F.lit(contract_version),
            ),
            256,
        ),
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually when a clean structural rebuild is required."
    )

if not spark.catalog.tableExists(quarantine_table):
    raise RuntimeError(
        f"Required quarantine target does not exist: {quarantine_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

target_columns = set(spark.table(quarantine_table).columns)
incoming_columns = set(quarantine_batch_df.columns)

if target_columns != incoming_columns:
    raise AssertionError(
        "Quarantine target schema does not match the quality output. "
        f"Missing in target: {sorted(incoming_columns - target_columns)}; "
        f"extra in target: {sorted(target_columns - incoming_columns)}"
    )

(
    DeltaTable.forName(spark, quarantine_table)
    .alias("target")
    .merge(
        quarantine_batch_df.alias("source"),
        "target._quarantine_record_id = source._quarantine_record_id",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

quarantine_batch_count = (
    spark.table(quarantine_table)
    .filter(
        (F.col("_batch_id") == batch_id)
        & (F.col("_quality_contract_version") == contract_version)
    )
    .count()
)

if quarantine_batch_count != rejected_count:
    raise AssertionError(
        f"Quarantine count mismatch: expected {rejected_count}, "
        f"found {quarantine_batch_count}."
    )

print(
    f"✅ Quarantine persisted idempotently: "
    f"{quarantine_batch_count:,} rows for this batch and contract."
)


## 8. Normalize the valid Silver candidate schema

Only records that passed every rule reach this step. Column names are converted to a consistent `snake_case` contract, text is trimmed, numeric types are explicit, and analytics columns are derived. The original Bronze identity and lineage remain available for traceability.

In [0]:
silver_candidate_df = (
    valid_quality_df
    .select(
        F.col("_bronze_record_id").alias("transaction_line_id"),
        F.trim(F.col("InvoiceNo")).alias("invoice_no"),
        F.trim(F.col("StockCode")).alias("stock_code"),
        F.trim(F.col("Description")).alias("description"),
        F.col("Quantity").cast("long").alias("quantity"),
        F.col("InvoiceDate").cast("timestamp").alias("invoice_timestamp"),
        F.col("UnitPrice").cast("decimal(18,4)").alias("unit_price"),
        F.trim(F.col("CustomerID")).alias("customer_id"),
        F.trim(F.col("Country")).alias("country"),
        F.col("_record_hash").alias("source_record_hash"),
        F.col("_batch_id").alias("source_batch_id"),
        F.col("_source_file").alias("source_file"),
        F.col("_source_sheet").alias("source_sheet"),
        F.col("_source_row_number").alias("source_row_number"),
        F.col("_input_file_path").alias("input_file_path"),
        F.col("_bronze_ingested_at").alias("bronze_ingested_at"),
        F.col("_quality_contract_version").alias("quality_contract_version"),
        F.col("_quality_checked_at").alias("quality_checked_at"),
    )
    .withColumn("sales_amount", (F.col("quantity") * F.col("unit_price")).cast("decimal(20,4)"))
    .withColumn("invoice_date", F.to_date("invoice_timestamp"))
    .withColumn("invoice_year", F.year("invoice_timestamp"))
    .withColumn("invoice_month", F.month("invoice_timestamp"))
    .withColumn("silver_prepared_at", F.current_timestamp())
)

candidate_count = silver_candidate_df.count()
candidate_distinct_ids = silver_candidate_df.select("transaction_line_id").distinct().count()
if candidate_count != valid_count or candidate_count != candidate_distinct_ids:
    raise AssertionError(
        f"Silver candidate uniqueness failed: rows={candidate_count}, "
        f"valid={valid_count}, distinct IDs={candidate_distinct_ids}."
    )

silver_candidate_df.printSchema()
display(silver_candidate_df.limit(20))

## 9. Persist the batch-specific Silver candidate

The candidate is written to its own batch-specific Delta path with overwrite mode. This makes reruns deterministic without overwriting another batch. The next notebook reads this exact path and merges the records into the final Silver table.

In [0]:
(
    silver_candidate_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_candidate_path)
)

persisted_candidate_df = spark.read.format("delta").load(silver_candidate_path)
persisted_candidate_count = persisted_candidate_df.count()
if persisted_candidate_count != valid_count:
    raise AssertionError(
        f"Persisted candidate count mismatch: expected {valid_count}, "
        f"found {persisted_candidate_count}."
    )

print(f"✅ Silver candidate written: {silver_candidate_path}")
print(f"Candidate rows: {persisted_candidate_count:,}")

## 10. Upsert batch-level quality metrics

The metrics table is pre-created by `lab04_00_setup`. One row is maintained per `(batch_id, contract_version)` so reruns update the same reconciliation record rather than creating duplicates.


In [0]:
quality_metrics_df = (
    build_quality_metrics(quality_df)
    .withColumnRenamed("input_rows", "bronze_rows")
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("contract_version", F.lit(contract_version))
    .withColumn(
        "valid_percentage",
        F.round(
            F.col("valid_rows") / F.col("bronze_rows") * 100,
            4,
        ),
    )
    .withColumn(
        "rejected_percentage",
        F.round(
            F.col("rejected_rows") / F.col("bronze_rows") * 100,
            4,
        ),
    )
    .withColumn(
        "candidate_path",
        F.lit(silver_candidate_path),
    )
    .withColumn(
        "measured_at",
        F.current_timestamp(),
    )
)

if not spark.catalog.tableExists(quality_metrics_table):
    raise RuntimeError(
        f"Required quality-metrics target does not exist: "
        f"{quality_metrics_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

metrics_target_schema = {
    field.name: field.dataType.simpleString()
    for field in spark.table(quality_metrics_table).schema.fields
}

metrics_source_schema = {
    field.name: field.dataType.simpleString()
    for field in quality_metrics_df.schema.fields
}

if metrics_target_schema != metrics_source_schema:
    raise AssertionError(
        "Quality-metrics target schema does not match the generated metrics. "
        f"Target: {metrics_target_schema}; "
        f"Generated: {metrics_source_schema}"
    )

(
    DeltaTable.forName(spark, quality_metrics_table)
    .alias("target")
    .merge(
        quality_metrics_df.alias("source"),
        (
            "target.batch_id = source.batch_id "
            "AND target.contract_version = source.contract_version"
        ),
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

display(quality_metrics_df)


## 11. Final validation and evidence

This final cell verifies the three persistent outputs for the selected batch: the candidate Delta path, quarantine table, and quality metrics table.

In [0]:
metrics_key_count = (
    spark.table(quality_metrics_table)
    .filter(
        (F.col("batch_id") == batch_id)
        & (F.col("contract_version") == contract_version)
    )
    .count()
)

if metrics_key_count != 1:
    raise AssertionError(f"Expected one quality-metrics row, found {metrics_key_count}.")

validation_df = spark.createDataFrame(
    [
        ("bronze_rows", bronze_batch_count),
        ("valid_candidate_rows", persisted_candidate_count),
        ("quarantined_rows", quarantine_batch_count),
        ("duplicate_rows_rejected", duplicate_row_count),
        ("quality_metric_rows_for_key", metrics_key_count),
    ],
    ["validation", "result"],
)

display(validation_df)
print("✅ Silver-quality processing completed successfully.")

## 12. Completion checklist and next notebook

This notebook is complete when:

- every selected Bronze row is classified as `VALID` or `REJECTED`;
- the reconciliation check proves `Bronze = valid + rejected`;
- each rejected record has at least one explicit reason code;
- exact business duplicates are absent from the Silver candidate;
- quarantine and quality metrics are idempotent for the batch and contract;
- the batch-specific Silver candidate is readable from its Delta path.

Recommended screenshots: rule-failure summary, reconciliation counts, sample rejected rows with reason arrays, normalized candidate schema, and final validation table.

### Next notebook

Continue with **`lab04_04_silver_merge.ipynb`**. It will load the persisted candidate path, create the final Silver Delta table with a defined schema, perform an upsert with `MERGE`, and prove that rerunning the same batch does not change the row count.